1. Start Spark

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("PySpark Practice") \
    .getOrCreate()

In [2]:
from google.colab import files
files.upload()

Saving employees.csv to employees.csv
Saving logins.txt to logins.txt
Saving orders.json to orders.json
Saving sales.csv to sales.csv


{'employees.csv': b'emp_id,name,department,salary,city\r\n1,Rahul,IT,70000,Hyderabad\r\n2,Sneha,HR,60000,Bangalore\r\n3,Arjun,IT,75000,Chennai\r\n4,Priya,Finance,80000,Hyderabad\r\n5,Karan,IT,50000,Mumbai\r\n6,Amit,HR,58000,Delhi\r\n7,Meera,Finance,82000,Bangalore\r\n8,Ravi,IT,72000,Hyderabad\r\n9,Neha,HR,61000,Chennai\r\n10,Vikram,Finance,90000,Delhi\r\n11,Anita,IT,65000,Bangalore\r\n12,Manoj,HR,62000,Mumbai\r\n13,Divya,IT,77000,Hyderabad\r\n14,Sanjay,Finance,85000,Chennai\r\n15,Pooja,IT,69000,Bangalore\r\n16,Kunal,HR,61000,Delhi\r\n17,Sonal,Finance,88000,Mumbai\r\n18,Deepak,IT,73000,Hyderabad\r\n19,Ritu,HR,60000,Chennai\r\n20,Akash,Finance,91000,Delhi',
 'logins.txt': b'Rahul\r\nSneha\r\nArjun\r\nRahul\r\nPriya\r\nSneha\r\nRahul\r\nKaran\r\nArjun\r\nSneha\r\nRahul\r\nAmit\r\nPriya\r\nKaran\r\nSneha\r\nRahul\r\nMeera\r\nArjun\r\nSneha\r\nRahul\r\nKaran\r\nAmit\r\nPriya\r\nSneha\r\nRahul\r\nArjun',
 'orders.json': b'{\r\n  "orders": [\r\n    {"order_id":1,"customer":"Rahul","product":"

In [3]:
!ls

employees.csv  logins.txt  orders.json	sales.csv  sample_data


DATASET 1 — TXT (Login Logs)
Load TXT File

In [6]:
logins = spark.read.text("logins.txt")


1. Print all names

In [7]:
logins.show(truncate=False)

+-----+
|value|
+-----+
|Rahul|
|Sneha|
|Arjun|
|Rahul|
|Priya|
|Sneha|
|Rahul|
|Karan|
|Arjun|
|Sneha|
|Rahul|
|Amit |
|Priya|
|Karan|
|Sneha|
|Rahul|
|Meera|
|Arjun|
|Sneha|
|Rahul|
+-----+
only showing top 20 rows


2. Total login events

In [8]:
logins.count()

26

3. Unique users

In [9]:
logins.select("value").distinct().show()

+-----+
|value|
+-----+
|Meera|
|Sneha|
|Priya|
|Rahul|
|Arjun|
| Amit|
|Karan|
+-----+



4. Login count per user

In [10]:
login_count = logins.groupBy("value").count()
login_count.show()

+-----+-----+
|value|count|
+-----+-----+
|Meera|    1|
|Sneha|    6|
|Priya|    3|
|Rahul|    7|
|Arjun|    4|
| Amit|    2|
|Karan|    3|
+-----+-----+



5. Top 3 active users

In [11]:
login_count.orderBy(desc("count")).show(3)

+-----+-----+
|value|count|
+-----+-----+
|Rahul|    7|
|Sneha|    6|
|Arjun|    4|
+-----+-----+
only showing top 3 rows


6. Users logged in more than 4 times

In [12]:
login_count.filter(col("count") > 4).show()

+-----+-----+
|value|count|
+-----+-----+
|Sneha|    6|
|Rahul|    7|
+-----+-----+



7. Convert to dictionary

In [13]:
login_dict = {row['value']: row['count'] for row in login_count.collect()}
print(login_dict)

{'Meera': 1, 'Sneha': 6, 'Priya': 3, 'Rahul': 7, 'Arjun': 4, 'Amit': 2, 'Karan': 3}


DATASET 2 — Employees CSV
1. Load CSV

In [14]:
employees = spark.read.csv(
    "employees.csv",
    header=True,
    inferSchema=True
)
employees.show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|     1| Rahul|        IT| 70000|Hyderabad|
|     2| Sneha|        HR| 60000|Bangalore|
|     3| Arjun|        IT| 75000|  Chennai|
|     4| Priya|   Finance| 80000|Hyderabad|
|     5| Karan|        IT| 50000|   Mumbai|
|     6|  Amit|        HR| 58000|    Delhi|
|     7| Meera|   Finance| 82000|Bangalore|
|     8|  Ravi|        IT| 72000|Hyderabad|
|     9|  Neha|        HR| 61000|  Chennai|
|    10|Vikram|   Finance| 90000|    Delhi|
|    11| Anita|        IT| 65000|Bangalore|
|    12| Manoj|        HR| 62000|   Mumbai|
|    13| Divya|        IT| 77000|Hyderabad|
|    14|Sanjay|   Finance| 85000|  Chennai|
|    15| Pooja|        IT| 69000|Bangalore|
|    16| Kunal|        HR| 61000|    Delhi|
|    17| Sonal|   Finance| 88000|   Mumbai|
|    18|Deepak|        IT| 73000|Hyderabad|
|    19|  Ritu|        HR| 60000|  Chennai|
|    20| Akash|   Finance| 91000

2. Total employees

In [15]:
employees.count()

20

3. IT department employees

In [16]:
employees.filter(col("department") == "IT").show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|     1| Rahul|        IT| 70000|Hyderabad|
|     3| Arjun|        IT| 75000|  Chennai|
|     5| Karan|        IT| 50000|   Mumbai|
|     8|  Ravi|        IT| 72000|Hyderabad|
|    11| Anita|        IT| 65000|Bangalore|
|    13| Divya|        IT| 77000|Hyderabad|
|    15| Pooja|        IT| 69000|Bangalore|
|    18|Deepak|        IT| 73000|Hyderabad|
+------+------+----------+------+---------+



4. Salary > 75000

In [17]:
employees.filter(col("salary") > 75000).show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|     4| Priya|   Finance| 80000|Hyderabad|
|     7| Meera|   Finance| 82000|Bangalore|
|    10|Vikram|   Finance| 90000|    Delhi|
|    13| Divya|        IT| 77000|Hyderabad|
|    14|Sanjay|   Finance| 85000|  Chennai|
|    17| Sonal|   Finance| 88000|   Mumbai|
|    20| Akash|   Finance| 91000|    Delhi|
+------+------+----------+------+---------+



5. Average salary

In [18]:
employees.select(avg("salary")).show()

+-----------+
|avg(salary)|
+-----------+
|    71450.0|
+-----------+



6. Highest paid employee

In [19]:
employees.orderBy(desc("salary")).show(1)

+------+-----+----------+------+-----+
|emp_id| name|department|salary| city|
+------+-----+----------+------+-----+
|    20|Akash|   Finance| 91000|Delhi|
+------+-----+----------+------+-----+
only showing top 1 row


7. Lowest paid employee

In [20]:
employees.orderBy("salary").show(1)

+------+-----+----------+------+------+
|emp_id| name|department|salary|  city|
+------+-----+----------+------+------+
|     5|Karan|        IT| 50000|Mumbai|
+------+-----+----------+------+------+
only showing top 1 row


8. Employees per department

In [21]:
employees.groupBy("department").count().show()

+----------+-----+
|department|count|
+----------+-----+
|        HR|    6|
|   Finance|    6|
|        IT|    8|
+----------+-----+



9. Avg salary per department

In [22]:
employees.groupBy("department").agg(avg("salary")).show()

+----------+------------------+
|department|       avg(salary)|
+----------+------------------+
|        HR|60333.333333333336|
|   Finance|           86000.0|
|        IT|           68875.0|
+----------+------------------+



10. Employees per city

In [23]:
employees.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    4|
|  Chennai|    4|
|   Mumbai|    3|
|    Delhi|    4|
|Hyderabad|    5|
+---------+-----+



11. Top 5 salaries

In [24]:
employees.orderBy(desc("salary")).show(5)

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|    20| Akash|   Finance| 91000|    Delhi|
|    10|Vikram|   Finance| 90000|    Delhi|
|    17| Sonal|   Finance| 88000|   Mumbai|
|    14|Sanjay|   Finance| 85000|  Chennai|
|     7| Meera|   Finance| 82000|Bangalore|
+------+------+----------+------+---------+
only showing top 5 rows


12. Hyderabad employees salary > 70k

In [25]:
employees.filter(
    (col("city") == "Hyderabad") &
    (col("salary") > 70000)
).show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|     4| Priya|   Finance| 80000|Hyderabad|
|     8|  Ravi|        IT| 72000|Hyderabad|
|    13| Divya|        IT| 77000|Hyderabad|
|    18|Deepak|        IT| 73000|Hyderabad|
+------+------+----------+------+---------+



Load Sales Data

In [26]:
sales = spark.read.csv(
    "sales.csv",
    header=True,
    inferSchema=True
)
sales.show()

+-------+------+--------+--------+-----+
|sale_id|emp_id| product|quantity|price|
+-------+------+--------+--------+-----+
|      1|     1|  Laptop|       1|75000|
|      2|     2|   Mouse|       3|  500|
|      3|     3|Keyboard|       2| 1500|
|      4|     1| Monitor|       1|12000|
|      5|     4|  Laptop|       1|75000|
|      6|     3|   Mouse|       2|  500|
|      7|     5|Keyboard|       1| 1500|
|      8|     1|  Laptop|       1|75000|
|      9|     6|   Mouse|       4|  500|
|     10|     7| Monitor|       2|12000|
|     11|     8|Keyboard|       2| 1500|
|     12|     9|   Mouse|       3|  500|
|     13|    10|  Laptop|       1|75000|
|     14|    11| Monitor|       1|12000|
|     15|    12|Keyboard|       2| 1500|
|     16|    13|  Laptop|       1|75000|
|     17|    14|   Mouse|       3|  500|
|     18|    15| Monitor|       1|12000|
|     19|    16|Keyboard|       1| 1500|
|     20|    17|  Laptop|       1|75000|
+-------+------+--------+--------+-----+



1. Revenue per sale

In [27]:
sales = sales.withColumn(
    "revenue",
    col("quantity") * col("price")
)

sales.show()

+-------+------+--------+--------+-----+-------+
|sale_id|emp_id| product|quantity|price|revenue|
+-------+------+--------+--------+-----+-------+
|      1|     1|  Laptop|       1|75000|  75000|
|      2|     2|   Mouse|       3|  500|   1500|
|      3|     3|Keyboard|       2| 1500|   3000|
|      4|     1| Monitor|       1|12000|  12000|
|      5|     4|  Laptop|       1|75000|  75000|
|      6|     3|   Mouse|       2|  500|   1000|
|      7|     5|Keyboard|       1| 1500|   1500|
|      8|     1|  Laptop|       1|75000|  75000|
|      9|     6|   Mouse|       4|  500|   2000|
|     10|     7| Monitor|       2|12000|  24000|
|     11|     8|Keyboard|       2| 1500|   3000|
|     12|     9|   Mouse|       3|  500|   1500|
|     13|    10|  Laptop|       1|75000|  75000|
|     14|    11| Monitor|       1|12000|  12000|
|     15|    12|Keyboard|       2| 1500|   3000|
|     16|    13|  Laptop|       1|75000|  75000|
|     17|    14|   Mouse|       3|  500|   1500|
|     18|    15| Mon

2. Total revenue

In [28]:
sales.select(sum("revenue")).show()

+------------+
|sum(revenue)|
+------------+
|      529500|
+------------+



3. Revenue per product

In [29]:
sales.groupBy("product") \
     .agg(sum("revenue").alias("total_revenue")) \
     .show()

+--------+-------------+
| product|total_revenue|
+--------+-------------+
|  Laptop|       450000|
|   Mouse|         7500|
|Keyboard|        12000|
| Monitor|        60000|
+--------+-------------+



4. Quantity per product

In [30]:
sales.groupBy("product") \
     .agg(sum("quantity")).show()

+--------+-------------+
| product|sum(quantity)|
+--------+-------------+
|  Laptop|            6|
|   Mouse|           15|
|Keyboard|            8|
| Monitor|            5|
+--------+-------------+



5. Best selling product

In [31]:
sales.groupBy("product") \
     .agg(sum("quantity").alias("qty")) \
     .orderBy(desc("qty")) \
     .show(1)

+-------+---+
|product|qty|
+-------+---+
|  Mouse| 15|
+-------+---+
only showing top 1 row


6. Employee generating highest revenue

In [32]:
sales.groupBy("emp_id") \
     .agg(sum("revenue").alias("revenue")) \
     .orderBy(desc("revenue")) \
     .show(1)

+------+-------+
|emp_id|revenue|
+------+-------+
|     1| 162000|
+------+-------+
only showing top 1 row


7. Average sale value

In [33]:
sales.select(avg("revenue")).show()

+------------+
|avg(revenue)|
+------------+
|     26475.0|
+------------+



8. Products revenue > 100000

In [34]:
sales.groupBy("product") \
     .agg(sum("revenue").alias("revenue")) \
     .filter(col("revenue") > 100000) \
     .show()

+-------+-------+
|product|revenue|
+-------+-------+
| Laptop| 450000|
+-------+-------+



DATASET 4 — JSON Orders

In [40]:
orders = spark.read.option("multiline","true") \
                   .json("orders.json")
orders = orders.select(explode("orders").alias("data")).select("data.*")


2. Print all orders

In [39]:
orders.show()

+------+---------+--------+--------+--------+
|amount|     city|customer|order_id| product|
+------+---------+--------+--------+--------+
| 75000|Hyderabad|   Rahul|       1|  Laptop|
|  1500|Bangalore|   Sneha|       2|   Mouse|
|  3000|  Chennai|   Arjun|       3|Keyboard|
| 75000|Hyderabad|   Priya|       4|  Laptop|
| 12000|   Mumbai|   Karan|       5| Monitor|
|  1000|Hyderabad|   Rahul|       6|   Mouse|
| 75000|Bangalore|   Sneha|       7|  Laptop|
|  3000|  Chennai|   Arjun|       8|Keyboard|
|  2000|Hyderabad|   Priya|       9|   Mouse|
| 12000|Hyderabad|   Rahul|      10| Monitor|
+------+---------+--------+--------+--------+



3. Total orders

In [41]:
orders.count()

10

4. Total sales amount

In [42]:
orders.select(sum("amount")).show()

+-----------+
|sum(amount)|
+-----------+
|     259500|
+-----------+



5. Spending per customer

In [43]:
orders.groupBy("customer") \
      .agg(sum("amount")).show()

+--------+-----------+
|customer|sum(amount)|
+--------+-----------+
|   Sneha|      76500|
|   Priya|      77000|
|   Rahul|      88000|
|   Arjun|       6000|
|   Karan|      12000|
+--------+-----------+



6. Highest spending customer

In [44]:
orders.groupBy("customer") \
      .agg(sum("amount").alias("spent")) \
      .orderBy(desc("spent")) \
      .show(1)

+--------+-----+
|customer|spent|
+--------+-----+
|   Rahul|88000|
+--------+-----+
only showing top 1 row


7. Sales per product

In [45]:
orders.groupBy("product") \
      .agg(sum("amount")).show()

+--------+-----------+
| product|sum(amount)|
+--------+-----------+
|  Laptop|     225000|
|   Mouse|       4500|
|Keyboard|       6000|
| Monitor|      24000|
+--------+-----------+



8. Customers from Hyderabad

In [46]:
orders.filter(col("city") == "Hyderabad").show()

+------+---------+--------+--------+-------+
|amount|     city|customer|order_id|product|
+------+---------+--------+--------+-------+
| 75000|Hyderabad|   Rahul|       1| Laptop|
| 75000|Hyderabad|   Priya|       4| Laptop|
|  1000|Hyderabad|   Rahul|       6|  Mouse|
|  2000|Hyderabad|   Priya|       9|  Mouse|
| 12000|Hyderabad|   Rahul|      10|Monitor|
+------+---------+--------+--------+-------+



9. Orders amount > 10000

In [47]:
orders.filter(col("amount") > 10000).show()

+------+---------+--------+--------+-------+
|amount|     city|customer|order_id|product|
+------+---------+--------+--------+-------+
| 75000|Hyderabad|   Rahul|       1| Laptop|
| 75000|Hyderabad|   Priya|       4| Laptop|
| 12000|   Mumbai|   Karan|       5|Monitor|
| 75000|Bangalore|   Sneha|       7| Laptop|
| 12000|Hyderabad|   Rahul|      10|Monitor|
+------+---------+--------+--------+-------+



10. Orders per city

In [48]:
orders.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|  Chennai|    2|
|   Mumbai|    1|
|Hyderabad|    5|
+---------+-----+



1. Join Employees + Sales

In [49]:
joined = employees.join(sales, "emp_id")
joined.show()

+------+------+----------+------+---------+-------+--------+--------+-----+-------+
|emp_id|  name|department|salary|     city|sale_id| product|quantity|price|revenue|
+------+------+----------+------+---------+-------+--------+--------+-----+-------+
|     1| Rahul|        IT| 70000|Hyderabad|      8|  Laptop|       1|75000|  75000|
|     1| Rahul|        IT| 70000|Hyderabad|      4| Monitor|       1|12000|  12000|
|     1| Rahul|        IT| 70000|Hyderabad|      1|  Laptop|       1|75000|  75000|
|     2| Sneha|        HR| 60000|Bangalore|      2|   Mouse|       3|  500|   1500|
|     3| Arjun|        IT| 75000|  Chennai|      6|   Mouse|       2|  500|   1000|
|     3| Arjun|        IT| 75000|  Chennai|      3|Keyboard|       2| 1500|   3000|
|     4| Priya|   Finance| 80000|Hyderabad|      5|  Laptop|       1|75000|  75000|
|     5| Karan|        IT| 50000|   Mumbai|      7|Keyboard|       1| 1500|   1500|
|     6|  Amit|        HR| 58000|    Delhi|      9|   Mouse|       4|  500| 

2. Revenue per employee

In [50]:
employee_revenue = joined.groupBy("name") \
    .agg(sum("revenue").alias("total_revenue"))

employee_revenue.show()

+------+-------------+
|  name|total_revenue|
+------+-------------+
| Kunal|         1500|
| Sonal|        75000|
| Divya|        75000|
|  Ravi|         3000|
|Sanjay|         1500|
| Meera|        24000|
| Sneha|         1500|
| Priya|        75000|
|Vikram|        75000|
| Rahul|       162000|
| Anita|        12000|
| Manoj|         3000|
| Pooja|        12000|
| Arjun|         4000|
|  Amit|         2000|
|  Neha|         1500|
| Karan|         1500|
+------+-------------+



3. Top 5 employees by sales

In [51]:
employee_revenue.orderBy(desc("total_revenue")).show(5)

+------+-------------+
|  name|total_revenue|
+------+-------------+
| Rahul|       162000|
| Priya|        75000|
| Divya|        75000|
| Sonal|        75000|
|Vikram|        75000|
+------+-------------+
only showing top 5 rows


4. Department generating highest revenue

In [52]:
joined.groupBy("department") \
      .agg(sum("revenue").alias("dept_revenue")) \
      .orderBy(desc("dept_revenue")) \
      .show(1)

+----------+------------+
|department|dept_revenue|
+----------+------------+
|        IT|      269500|
+----------+------------+
only showing top 1 row


5. Save Final Report

In [53]:
employee_revenue.write \
    .mode("overwrite") \
    .option("header",True) \
    .csv("final_sales_report.csv")